# Workflow of this pipeline

1) **resave** `nd2` images as `tif`, splitting individual channels

2) create RS-FISH detection settings `Log.txt` file in Fiji (independent of this pipeline)

-> **detect spots** using RS-FISH producing:

    - `/.../detections` folder containing a `csv` with spot information for each provided `tif`
    - `merge.csv` combining all `csvs` in the `detections` folder

-> 2.1) (optionaly) correct chromatic shift using a reference registration file, producing a `*_shift-corrected.csv` file for each input file 

3) **visualise the detections**, creating a `/.../detections/vis` folder containing `png` max projections of images in the spot file (eg. `merge.csv`)

4) **segment nuclei** with cellpose, using `tif`s and a cellpose classifyer (default or costum trained) producing:
    - `/.../segmentation` folder containing 2D / 3D segmentation masks in `.npy` and `.tif.` format
    - `/.../segmentation/vis` folder containing 2D / max projection `.png` visualizations of the segmentation
    
4) .1 **filter spots not in nuclei** and calculate sensitivity based on spots from (3) and segmentation masks from (4)
5) spots in 2 channels are matched based on the closest neighbour and **spot distances calculated**
6) **calculate distances** between paired spots

In [ ]:
import papermill as pm
import pandas as pd
from multiprocessing import Pool
import concurrent.futures
import queue
import os
from pathlib import Path
from datetime import datetime

In [ ]:
def run_notebook(parameters,notebook_to_run,parameters_common={}):
    
    # change to directory where the notebook is (resolve relative imports)
    os.chdir(Path(notebook_to_run).absolute().parent)
    
    # run notebook
    for parameters_spec in parameters_list:
        parameters = {**parameters_common, **parameters_spec}

        pm.execute_notebook(
           notebook_to_run,
            "/home/stumberger/image-analyis-recipes/alignment/correct_chromatic_aberration_for_tables_gs_copy.ipynb",
#            '/dev/null',
           parameters=parameters)

# 0) Create projections for visualization

In [ ]:
parameters_list = [
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run0_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run1_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run0_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run1_sd/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_CTRL/raw"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_MNX1/raw"}
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/visualization/nd2_make_projections.ipynb"

run_notebook(parameters_list,notebook_to_run)

# 1) Resave .nd2 to tif 

In [ ]:
parameters_list = [
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run0_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run1_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run0_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run1_sd/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_CTRL/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_MNX1/"},
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/resave/resave_nd2_as_tiff.ipynb"

run_notebook(parameters_list,notebook_to_run)

# 2) Spot detection

In [ ]:
parameters_list = [
#     {"path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run0_sd/"},
#     {"path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run1_sd/"},
#     {"path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run0_sd/"},
#     {"path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run1_sd/"},
    {"path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_CTRL/"},
    {"path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_MNX1/"},
]

parameters_common = {"channels": [1,2]}

notebook_to_run = "/home/stumberger/image-analyis-recipes/spot-detection/RS-FISH_spot_detection.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 3) Correct chromatic shift

In [ ]:
parameters_list = [
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run0_sd/detections/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run1_sd/detections/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run0_sd/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run1_sd/detections/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_CTRL/detections/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_MNX1/detections/"},
]

parameters_common = {"channels": [1,2],
                     "pixel_size": [0.3,0.13,0.13],
                     "channel_aliases": {'405 CSU-W1': '405-CSU-W1',
                                         '488 CSU-W1': '488-CSU-W1',
                                         '561 CSU-W1': 1,
                                         '640 CSU-W1': 2},
                     "transforms_path": "/data/agl_data/NanoFISH/Gabi/tools/sd_chrom_shift_regisitration.json"
}

notebook_to_run = "/home/stumberger/image-analyis-recipes/alignment/correct_chromatic_aberration_for_tables_gs.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 4) Segment cells

In [ ]:
parameters_common = {
    "model": "nuclei"}

parameters_list = [
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run0_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run1_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run0_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run1_sd/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_CTRL/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_MNX1/"},
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/segmentation/cellpose_segmentation_3d.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 4.1) Add segmentation info to spots

In [ ]:
parameters_common = {
    "filter": True,
    "mask_ending": "_seg"}

parameters_list = [
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run0_sd/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run1_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run0_sd/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run1_sd/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_CTRL/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_MNX1/"},
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/measurement/assign_spots_to_cell.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 5) Calculate distances

In [ ]:
parameters_common = {
    "rel_spot_path": "/detections/merge_filtered.csv"} #spot path relative to upper folder

parameters_list = [
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run0_sd/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run1_sd/"},
#     {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run0_sd/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run1_sd/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_CTRL/"},
    {"in_path": "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_MNX1/"},
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/measurement/paired_spot_distances_2_channels.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 6) Join csv files

In [ ]:
wd = "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/"

csv_files = [
    "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run0_sd/",
    "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_CTRL/20231013_run1_sd/",
#   "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run0_sd/",
    "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/01_20231013_GDM1_AHI1_MNX1/20231013_run1_sd/",
    "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_CTRL/",
    "/data/agl_data/NanoFISH/Clemens/0_Christoph_Plass/GDM1_AHI1_MNX1_NewCrtlChr6/02_20231025_GDM1_AHI1_MNX1/",
]

# Initialize an empty list to store DataFrames
dataframes = []

# Read and store each CSV file as a DataFrame
for file in csv_files:
    df = pd.read_csv(f"{file}/distances.csv")
    dataframes.append(df)

# Join the DataFrames using Pandas (e.g., concatenate them vertically)
joined_dataframe = pd.concat(dataframes, ignore_index=True)

# Save the joined DataFrame to a new CSV file
time =  datetime.now().strftime("%Y-%m-%d-%H-%M")
joined_dataframe.to_csv(f'{wd}/all_distances_{time}.csv', index=False)